In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# EDA

In [2]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print("=== DATA SHAPE & COLUMNS ===")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}\n")

print("=== MISSING VALUES ===")
print(df.isnull().sum())
print("\n=== SAMPLE HEAD ===")
print(df.head(2))

if 'answer' in df.columns:
    print("\n=== TARGET DISTRIBUTION ===")
    print(df['answer'].value_counts(dropna=False, normalize=True) * 100)

=== DATA SHAPE & COLUMNS ===
Shape: (2000, 8)
Columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

=== MISSING VALUES ===
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

=== SAMPLE HEAD ===
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   C  \
0  Martin Heidegger does not believe in the exist...   
1  Accelerator-based light-ion fusion is a techni...   

                                                   

In [3]:
import matplotlib.pyplot as plt

option_cols = ['A', 'B', 'C', 'D', 'E']
prompt_col = 'prompt'
answer_col = 'answer'


if prompt_col:
    df['prompt_word_len'] = df[prompt_col].astype(str).apply(lambda x: len(x.split()))
if answer_col:
    df['answer_word_len'] = df[answer_col].astype(str).apply(lambda x: len(x.split()))

for col in option_cols:
    df[f'{col}_word_len'] = df[col].astype(str).apply(lambda x: len(x.split()))

df['total_word_len'] = df[[c for c in df.columns if 'word_len' in c]].sum(axis=1)

print("=== LENGTH STATS (WORD COUNTS) ===")
print(df[[c for c in df.columns if 'word_len' in c]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

=== LENGTH STATS (WORD COUNTS) ===
       prompt_word_len  answer_word_len   A_word_len   B_word_len  C_word_len  \
count       2000.00000           2000.0  2000.000000  2000.000000  2000.00000   
mean          18.14650              1.0    26.146000    26.516000    26.54950   
std            6.78189              0.0    17.249651    18.653893    17.20987   
min            3.00000              1.0     1.000000     1.000000     1.00000   
50%           17.00000              1.0    22.000000    23.000000    24.00000   
90%           27.00000              1.0    49.000000    49.000000    48.00000   
95%           31.00000              1.0    62.000000    62.000000    60.00000   
99%           37.00000              1.0    78.000000    84.000000    81.00000   
max           51.00000              1.0    81.000000   118.000000    82.00000   

        D_word_len   E_word_len  total_word_len  
count  2000.000000  2000.000000     2000.000000  
mean     25.999500    26.187000      150.544500  
std 

In [4]:
for col in option_cols:
    df[f'{col}_char_len'] = df[col].astype(str).apply(len)

if 'answer' in df.columns:
    def get_correct_option_len(row):
        ans = str(row['answer']).strip()
        for col in option_cols:
            if ans == col:
                return len(str(row[col]))
        return None

    df['correct_option_char_len'] = df.apply(get_correct_option_len, axis=1)
    
    print("=== OPTION LENGTH COMPARISON ===")
    print(f"Mean Correct Option Length: {df['correct_option_char_len'].mean():.2f} chars")
    print("Mean Option Lengths by Column:")
    for col in option_cols:
        print(f"  {col}: {df[f'{col}_char_len'].mean():.2f} chars")

=== OPTION LENGTH COMPARISON ===
Mean Correct Option Length: 181.88 chars
Mean Option Lengths by Column:
  A: 164.09 chars
  B: 166.62 chars
  C: 167.23 chars
  D: 162.56 chars
  E: 164.23 chars


In [5]:
if prompt_col:
    exact_dup_prompts = df[prompt_col].duplicated().sum()
    print("=== DUPLICATE ANALYSIS ===")
    print(f"Exact Duplicate Prompts: {exact_dup_prompts} ({(exact_dup_prompts/len(df))*100:.2f}%)")

def check_duplicate_options(row):
    opts = [str(row[col]).strip() for col in option_cols]
    return len(opts) != len(set(opts))

df['has_duplicate_options'] = df.apply(check_duplicate_options, axis=1)
print(f"Questions with duplicate options: {df['has_duplicate_options'].sum()}")

=== DUPLICATE ANALYSIS ===
Exact Duplicate Prompts: 242 (12.10%)
Questions with duplicate options: 0


In [6]:
import re

def detect_patterns(text):
    text = str(text)
    has_latex = bool(re.search(r'\$|\\frac|\\sqrt|\\alpha|\\beta', text))
    has_code = bool(re.search(r'def\s+|class\s+|import\s+|for\s+.*in|#include', text))
    has_math_symbols = bool(re.search(r'[=+\-*/^<>≤≥≠±]', text))
    return pd.Series([has_latex, has_code, has_math_symbols], index=['has_latex', 'has_code', 'has_math_symbols'])

if prompt_col:
    pattern_df = df[prompt_col].apply(detect_patterns)
    print("=== DOMAIN & SYNTAX PATTERNS ===")
    print(pattern_df.mean() * 100)

=== DOMAIN & SYNTAX PATTERNS ===
has_latex            0.00
has_code             8.35
has_math_symbols    22.85
dtype: float64


In [7]:
import transformers
from huggingface_hub import HfApi

print(f"=== TRANSFORMERS VERSION ===")
print(f"Version: {transformers.__version__}\n")

candidates = [
    "roberta-base",
    "roberta-large",
    "microsoft/deberta-v3-base",
    "microsoft/deberta-v3-small",
    "bert-base-uncased",
    "google/electra-base-discriminator"
]

print("=== CHECKING HF MODEL ACCESSIBILITY ===")
api = HfApi()

for model_id in candidates:
    try:
        info = api.model_info(model_id)
        print(f"✅ FOUND: '{model_id}'")
    except Exception as e:
        print(f"❌ NOT FOUND / ERROR: '{model_id}' -> {e}")

=== TRANSFORMERS VERSION ===
Version: 5.0.0

=== CHECKING HF MODEL ACCESSIBILITY ===
✅ FOUND: 'roberta-base'
✅ FOUND: 'roberta-large'
✅ FOUND: 'microsoft/deberta-v3-base'
✅ FOUND: 'microsoft/deberta-v3-small'
✅ FOUND: 'bert-base-uncased'
✅ FOUND: 'google/electra-base-discriminator'


In [8]:
import os
import torch
import wandb
from dataclasses import dataclass
from typing import Optional, Union
from torch.utils.data import DataLoader
from transformers import (
AutoTokenizer, 
AutoModelForMultipleChoice, 
TrainingArguments, 
Trainer,
PreTrainedTokenizerBase
)
from transformers.tokenization_utils_base import PaddingStrategy
from datasets import Dataset
from sklearn.model_selection import GroupKFold
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WB_KEY")
wandb.login(key=secret_value_0)

wandb.init(
    project="23f2002267-t22026", 
    name="roberta-base-baseline",
    reinit="finish_previous"
)

# global configuration
MODEL_NAME = "roberta-base"
MAX_LEN = 256
option_cols = ['A', 'B', 'C', 'D', 'E']
label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
id2label = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: setting up run 6talnbh5
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260729_122849-6talnbh5
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run roberta-base-baseline
wandb: ⭐️ View project at https://wandb.ai/23f2002267-dl-genai-project/23f2002267-t22026
wandb: 🚀 View run at https://wandb.ai/23f2002267-dl-genai-project/23f2002267-t22026/runs/6talnbh5


PyTorch Version: 2.10.0+cu128
CUDA Available: True


In [9]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
train_df['label'] = train_df['answer'].map(label2id)

print(f"Train Dataset Shape: {train_df.shape}")

# initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# preprocessing function for MCQ formatting
def preprocess_function(examples):
    first_sentences = [[f"Question: {prompt}"] * 5 for prompt in examples['prompt']]
    second_sentences = [
        [f"Option {opt}: {examples[opt][i]}" for opt in option_cols] 
        for i in range(len(examples['prompt']))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=MAX_LEN,
        padding=False
    )

    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

# apply 5-Fold GroupKFold based on 'prompt' to prevent duplicate prompt leakage
gkf = GroupKFold(n_splits=5)
train_df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=train_df['prompt'])):
    train_df.loc[val_idx, 'fold'] = fold

# select fold 0 for training/validation split
fold_0_train = train_df[train_df['fold'] != 0].reset_index(drop=True)
fold_0_val = train_df[train_df['fold'] == 0].reset_index(drop=True)

train_dataset = Dataset.from_pandas(fold_0_train).map(preprocess_function, batched=True)
val_dataset = Dataset.from_pandas(fold_0_val).map(preprocess_function, batched=True)

print(f"Fold 0 Train Samples: {len(train_dataset)} | Val Samples: {len(val_dataset)}")

Train Dataset Shape: (2000, 9)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Fold 0 Train Samples: 1600 | Val Samples: 400


In [10]:
@dataclass
class RobustDataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        has_labels = "label" in features[0] or "labels" in features[0]
        if has_labels:
            label_name = "label" if "label" in features[0] else "labels"
            labels = [feature.pop(label_name) for feature in features]
        
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        # keep only model tensor input keys
        model_inputs = ["input_ids", "attention_mask", "token_type_ids"]
        
        flattened_features = [
            [{k: feature[k][i] for k in feature if k in model_inputs} for i in range(num_choices)]
            for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if has_labels:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)
            
        return batch

print("Data Collator successfully initialized.")

Data Collator successfully initialized.


In [11]:
def compute_metrics(eval_predictions):
    predictions, labels = eval_predictions
    preds = np.argmax(predictions, axis=1)
    acc = (preds == labels).mean()
    return {"accuracy": acc}

# load pre-trained RoBERTa model
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_steps=50,
    max_grad_norm=1.0,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=20,
    report_to="wandb",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer, 
    data_collator=RobustDataCollatorForMultipleChoice(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

# start model fine tuning
trainer.train()
wandb.finish()

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension

Epoch,Training Loss,Validation Loss,Accuracy
1,5.536273,2.300865,0.680000
2,2.286646,0.857851,0.885000
3,1.593015,0.645468,0.920000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [12]:
# evaluation
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Load test dataset and sample submission
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(f"Test Set Shape: {test_df.shape}")

# Preprocess test set
def preprocess_test_function(examples):
    first_sentences = [[f"Question: {prompt}"] * 5 for prompt in examples['prompt']]
    second_sentences = [
        [f"Option {opt}: {examples[opt][i]}" for opt in option_cols] 
        for i in range(len(examples['prompt']))
    ]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized_examples = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=MAX_LEN,
        padding=False
    )

    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

# DataFrame to Dataset and format tensor columns
test_dataset = Dataset.from_pandas(test_df).map(preprocess_test_function, batched=True)
model_input_cols = [c for c in ["input_ids", "attention_mask", "token_type_ids"] if c in test_dataset.column_names]
test_dataset.set_format(type="python", columns=model_input_cols)

# DataLoader
collator = RobustDataCollatorForMultipleChoice(tokenizer=tokenizer)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collator)

# forward pass on test set
all_logits = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        all_logits.append(outputs.logits.cpu().numpy())

logits = np.concatenate(all_logits, axis=0)  # Shape: (500, 5)

# MAP@3 submission format
top3_indices = np.argsort(logits, axis=1)[:, ::-1][:, :3]
top3_predictions = [
    " ".join([id2label[idx] for idx in row]) 
    for row in top3_indices
]

id_col_name = sample_sub.columns[0]      # 'ID'
target_col_name = sample_sub.columns[1]  # 'Prediction'

submission_df = pd.DataFrame({
    id_col_name: test_df['id'],
    target_col_name: top3_predictions
})

submission_df.to_csv("submission.csv", index=False)

print("\n=== MAP@3 SUBMISSION HEAD ===")
print(submission_df.head(10))

Test Set Shape: (500, 7)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]


=== MAP@3 SUBMISSION HEAD ===
   ID Prediction
0   1      A D C
1   2      B D C
2   3      B E A
3   4      C D E
4   5      C A D
5   6      D B C
6   7      E C D
7   8      B E A
8   9      C D E
9  10      B C E


In [13]:
print("\n=== TOP PREDICTION PATTERNS ===")
print(submission_df[target_col_name].value_counts().head(10))


=== TOP PREDICTION PATTERNS ===
Prediction
D B C    23
C B A    22
C A D    21
B E C    20
D B E    18
C D B    18
E B C    15
C B E    14
A D C    14
B C D    14
Name: count, dtype: int64
